In [ ]:
import tensorflow as tf
import numpy as np

x_raw = [[1, 2, 1, 1],
          [2, 1, 3, 2],
          [3, 1, 3, 4],
          [4, 1, 5, 5],
          [1, 7, 5, 5],
          [1, 2, 5, 6],
          [1, 6, 6, 6],
          [1, 7, 7, 7]]
y_raw = [[0, 0, 1],
          [0, 0, 1],
          [0, 0, 1],
          [0, 1, 0],
          [0, 1, 0],
          [0, 1, 0],
          [1, 0, 0],
          [1, 0, 0]]

X = tf.Variable(x_raw, dtype=tf.float32)
Y = tf.Variable(y_raw, dtype=tf.float32)

tf.random.set_seed(1)
W = tf.Variable(tf.random.normal([X.shape[1], Y.shape[1]]))
b = tf.Variable(tf.random.normal([Y.shape[1]]))

## softmax_cross_entropy_with_logits 함수 사용

두 가지 방법으로 cost를 구할 수 있다. output은 같다.

In [ ]:
# 이전에 했던 코드
# hypothesis = tf.nn.softmax(tf.matmul(X, W) + b)
# cost = tf.reduce_mean(-tf.reduce_sum(Y * tf.math.log(hypothesis), axis=1))

# >>> <tf.Tensor: shape=(), dtype=float32, numpy=1.3499713>

In [ ]:
logits = tf.matmul(X, W) + b
cost_i = tf.nn.softmax_cross_entropy_with_logits(logits=logits, labels=Y)
cost = tf.reduce_mean(cost_i)

# >>> <tf.Tensor: shape=(), dtype=float32, numpy=1.3499713>

# Animal Classification

## 1. 데이터 로드

In [21]:
import tensorflow as tf
import numpy as np

data = np.loadtxt('https://raw.githubusercontent.com/hunkim/DeepLearningZeroToAll/master/data-04-zoo.csv', delimiter=',', dtype=np.float32)
print(data.shape)

X = data[:, :-1]
Y = data[:, -1]
print(X.shape, Y.shape)

(101, 17)
(101, 16) (101,)


## 2. 데이터 전처리

In [22]:
nb_classes = len(set(Y))
print(set(Y), nb_classes)

{0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0} 7


레이블을 개수를 확인해보면 0부터 6까지 7종류인 것을 볼 수 있다. 레이블을 원핫인코딩 해준다.

In [23]:
print(Y[0])
Y_one_hot = tf.one_hot(Y, nb_classes)
print(Y_one_hot[0])
print(Y_one_hot.shape)

0.0
tf.Tensor([1. 0. 0. 0. 0. 0. 0.], shape=(7,), dtype=float32)
(101, 7)


onehot인코딩 전후를 비교해보면 레이블이 0인 샘플은 0번 인덱스만 1이되고 나머지는 0으로 레이블이 0인 것을 확인할 수 있다.

강의에서는 reshape을 안하면 에러가 난다고 했는데 텐서플로 2에서는 reshape을 안해도 되는 것 같다.

## 3. 학습

In [24]:
# W, b 정의
features = X.shape[1]
tf.random.set_seed(1)

W = tf.Variable(tf.random.normal([features, nb_classes]))  # (16, 7)
b = tf.Variable(tf.random.normal([nb_classes]))            

In [27]:
# cost 정의
logits = tf.matmul(X, W) + b
cost_i = tf.nn.softmax_cross_entropy_with_logits(logits=logits, 
                                                 labels=Y_one_hot)
cost = tf.reduce_mean(cost_i)

In [ ]:
epochs = 2000
learning_rate = 0.1
for step in range(epochs):
    with tf.GradientTape() as tape:
        logits = tf.matmul(X, W) + b
        hypothesis = tf.nn.softmax(logits)
        cost_i = tf.nn.softmax_cross_entropy_with_logits(logits=logits, 
                                                        labels=Y_one_hot)
        cost = tf.reduce_mean(cost_i)
    prediction = tf.argmax(hypothesis, 1)
    correct_prediction = tf.equal(prediction, Y)
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))

    W_grad, b_grad = tape.gradient(cost, [W, b])
    
    W.assign_sub(learning_rate * W_grad)
    b.assign_sub(learning_rate * b_grad)
    if step % 200 == 0:
        print(f"{step:>4} {cost.numpy():>5.5f} {accuracy}")

## 4. 결과 확인

In [58]:
y_pred = tf.nn.softmax(tf.matmul(X, W) + b)    # 학습된 W와 b를 가지고 예측값을 만든다.
y_pred = tf.argmax(y_pred, axis=1)             # 가장 큰 확률의 인덱스를 반환한다.

# accuracy 
# tf 이용
cor = tf.equal(y_pred, Y)
accuracy = tf.reduce_mean(tf.cast(cor, tf.float32)).numpy()
print('Accuracy :', accuracy)

# 순정 파이썬
# print('Accuracy :', sum(Y == y_pred.numpy()) / len(Y))  # 실제 Y와 비교해서 맞은 비율 (accuracy)

Accuracy : 1.0


## 전체 코드

In [65]:
import tensorflow as tf
import numpy as np

data = np.loadtxt('https://raw.githubusercontent.com/hunkim/DeepLearningZeroToAll/master/data-04-zoo.csv', delimiter=',', dtype=np.float32)

X = data[:, :-1]
Y = data[:, -1]

nb_classes = len(set(Y))
Y_one_hot = tf.one_hot(Y, nb_classes)

features = X.shape[1]
tf.random.set_seed(1)

W = tf.Variable(tf.random.normal([features, nb_classes]))  # (16, 7)
b = tf.Variable(tf.random.normal([nb_classes]))            

epochs = 2000
learning_rate = 0.1
for step in range(epochs):
    with tf.GradientTape() as tape:
        logits = tf.matmul(X, W) + b
        hypothesis = tf.nn.softmax(logits)
        cost_i = tf.nn.softmax_cross_entropy_with_logits(logits=logits, 
                                                        labels=Y_one_hot)
        cost = tf.reduce_mean(cost_i)
    prediction = tf.argmax(hypothesis, 1)
    correct_prediction = tf.equal(prediction, Y)
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))

    W_grad, b_grad = tape.gradient(cost, [W, b])
    
    W.assign_sub(learning_rate * W_grad)
    b.assign_sub(learning_rate * b_grad)
    if step % 200 == 0:
        print(f"{step:>4} cost :{cost.numpy():>5.5f} accuracy :{accuracy:>5.4f}")

y_pred = tf.nn.softmax(tf.matmul(X, W) + b)    # 학습된 W와 b를 가지고 예측값을 만든다.
y_pred = tf.argmax(y_pred, axis=1)             # 가장 큰 확률의 인덱스를 반환한다.

   0 cost :4.23807 accuracy :0.1287
 200 cost :0.44560 accuracy :0.8713
 400 cost :0.26569 accuracy :0.9208
 600 cost :0.18789 accuracy :0.9406
 800 cost :0.14415 accuracy :0.9802
1000 cost :0.11620 accuracy :0.9802
1200 cost :0.09695 accuracy :1.0000
1400 cost :0.08302 accuracy :1.0000
1600 cost :0.07252 accuracy :1.0000
1800 cost :0.06437 accuracy :1.0000
Accuracy : 1.0
